<a href="https://colab.research.google.com/github/hardik-05/Fid-Case-Study/blob/main/spark_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Test')

Test


In [1]:
%pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840625 sha256=da5ea8e242514d969170ae951a6dd8de672ffcf91e4ec30b165f97579d3c0c49
  Stored in directory: /root/.cache/pip/wheels/1b/3a/92/28b93e2fbfdbb07509ca4d6f50c5e407f48dce4ddbda69a4ab
Successfully built pyspark


In [3]:
from pyspark import SparkContext, SparkConf

configuration = SparkConf().setAppName('app').setMaster('local')
sc = SparkContext.getOrCreate(conf=configuration)



In [4]:
temps = [10,3,-5,25,1,9,29,-10,5]

rdd = sc.parallelize(temps)
rdd.collect()


[10, 3, -5, 25, 1, 9, 29, -10, 5]

In [8]:
rdd2 = rdd.map(lambda x: x+273.15).collect()
print(rdd2)

[283.15, 276.15, 268.15, 298.15, 274.15, 282.15, 302.15, 263.15, 278.15]


In [9]:
sum_func = lambda x,y: x+y
rdd3 = sc.parallelize(rdd2)
total_temps = rdd3.reduce(sum_func)
print(total_temps)

2525.3500000000004


Shakespear Data


In [10]:
import sys, re
from pyspark import SparkContext, SparkConf

configuration = SparkConf().setAppName('word counter').setMaster('local')
sc = SparkContext.getOrCreate(conf=configuration)


In [11]:
#wordcounts = sc.textFile('shakespeare.txt')
wordcounts = sc.textFile("file:///content/shakespeare.txt")\
              .filter(lambda line: len(line) > 0)\
              .flatMap(lambda line : re.split('\W+', line))\
              .filter(lambda word : len(word) > 0)\
              .map(lambda word : (word.lower(),1))\
              .reduceByKey(lambda x,y : x+y)\
              .map(lambda x: (x[1],x[0]))\
              .sortByKey(ascending = False).persist()

topN = lambda x: wordcounts.take(x)
topN(10)


[(30212, 'the'),
 (28468, 'and'),
 (23956, 'i'),
 (21130, 'to'),
 (18823, 'of'),
 (16350, 'a'),
 (14693, 'you'),
 (13198, 'my'),
 (12399, 'in'),
 (12251, 'that')]

In [12]:
#wordcounts = sc.textFile('shakespeare.txt')
stop_words = ['the','and','i','to','of','a','you','my','in','that']
wordcounts = sc.textFile("file:///content/shakespeare.txt")\
              .filter(lambda line: len(line) > 0)\
              .flatMap(lambda line : re.split('\W+', line))\
              .filter(lambda word : len(word) > 0)\
              .map(lambda word : (word.lower(),1))\
              .filter(lambda x: x[0] not in stop_words)\
              .reduceByKey(lambda x,y : x+y)\
              .map(lambda x: (x[1],x[0]))\
              .sortByKey(ascending = False).persist()

topN = lambda x: wordcounts.take(x)
topN(10)


[(9908, 'is'),
 (9083, 'not'),
 (8956, 'd'),
 (8539, 'with'),
 (8420, 's'),
 (8303, 'for'),
 (8284, 'me'),
 (8240, 'it'),
 (7586, 'his'),
 (7411, 'be')]

In [16]:
#wordcounts = sc.textFile('shakespeare.txt')
stop_words = sc.textFile("file:///content/stopwords.txt")\
              .filter(lambda word: word.strip() != "").collect()
#print(stop_words)
wordcounts = sc.textFile("file:///content/shakespeare.txt")\
              .filter(lambda line: len(line) > 0)\
              .flatMap(lambda line : re.split('\W+', line))\
              .filter(lambda word : len(word) > 0)\
              .map(lambda word : (word.lower(),1))\
              .filter(lambda x: x[0] not in stop_words)\
              .reduceByKey(lambda x,y : x+y)\
              .map(lambda x: (x[1],x[0]))\
              .sortByKey(ascending = False).persist()

topN = lambda x: wordcounts.take(x)
topN(10)


[(5332, 'will'),
 (3210, 'lord'),
 (3152, 'king'),
 (3029, 'now'),
 (3004, 'sir'),
 (2647, 'come'),
 (2509, 'let'),
 (2501, 'll'),
 (2491, 'here'),
 (2481, 'enter')]

In [17]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk_stop_words = stopwords.words('english')
rdd = sc.parallelize(nltk_stop_words)
stop_words = rdd.filter(lambda word: word.strip() != "").collect()
#print(stop_words)

wordcounts = sc.textFile("file:///content/shakespeare.txt")\
              .filter(lambda line: len(line) > 0)\
              .flatMap(lambda line : re.split('\W+', line))\
              .filter(lambda word : len(word) > 0)\
              .map(lambda word : (word.lower(),1))\
              .filter(lambda x: x[0] not in stop_words)\
              .reduceByKey(lambda x,y : x+y)\
              .map(lambda x: (x[1],x[0]))\
              .sortByKey(ascending = False).persist()

topN = lambda x: wordcounts.take(x)
topN(10)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


[(5910, 'thou'),
 (4361, 'thy'),
 (3858, 'shall'),
 (3401, 'thee'),
 (3210, 'lord'),
 (3152, 'king'),
 (3004, 'sir'),
 (2991, 'good'),
 (2647, 'come'),
 (2509, 'let')]

Instructor

In [18]:
stop_words_2 = sc.textFile("file:///content/stopwords.txt")\
              .flatMap(lambda line: re.split('\W+',line))\
              .collect()
print(stop_words_2)

wordcounts = sc.textFile("file:///content/shakespeare.txt")\
              .filter(lambda line: len(line) > 0)\
              .flatMap(lambda line : re.split('\W+', line))\
              .filter(lambda word : len(word) > 0)\
              .map(lambda word : (word.lower(),1))\
              .filter(lambda x: x[0] not in stop_words_2)\
              .reduceByKey(lambda x,y : x+y)\
              .map(lambda x: (x[1],x[0]))\
              .sortByKey(ascending = False).persist()

topN = lambda x: wordcounts.take(x)
topN(10)


['x', 'y', 'your', 'yours', 'yourself', 'yourselves', 'you', 'yond', 'yonder', 'yon', 'ye', 'yet', 'z', 'zillion', 'j', 'u', 'umpteen', 'usually', 'us', 'username', 'uponed', 'upons', 'uponing', 'upon', 'ups', 'upping', 'upped', 'up', 'unto', 'until', 'unless', 'unlike', 'unliker', 'unlikest', 'under', 'underneath', 'use', 'used', 'usedest', 'r', 'rath', 'rather', 'rathest', 'rathe', 're', 'relate', 'related', 'relatively', 'regarding', 'really', 'res', 'respecting', 'respectively', 'q', 'quite', 'que', 'qua', 'n', 'neither', 'neaths', 'neath', 'nethe', 'nethermost', 'necessary', 'necessariest', 'necessarier', 'never', 'nevertheless', 'nigh', 'nighest', 'nigher', 'nine', 'noone', 'nobody', 'nobodies', 'nowhere', 'nowheres', 'no', 'noes', 'nor', 'nos', 'no', 'one', 'none', 'not', 'notwithstanding', 'nothings', 'nothing', 'nathless', 'natheless', 't', 'ten', 'tills', 'till', 'tilled', 'tilling', 'to', 'towards', 'toward', 'towardest', 'towarder', 'together', 'too', 'thy', 'thyself', 'thu

[(5332, 'will'),
 (3210, 'lord'),
 (3152, 'king'),
 (3029, 'now'),
 (3004, 'sir'),
 (2509, 'let'),
 (2501, 'll'),
 (2491, 'here'),
 (2481, 'enter'),
 (2433, 'love')]

Spark SQL

In [20]:
from pyspark import SparkContext,SparkConf, SQLContext
from pyspark import *
from pyspark import Row
from pyspark.sql import SparkSession


In [25]:
from pyspark import SparkContext, SparkConf
configuration = SparkConf().setAppName('states').setMaster('local')
sc = SparkContext.getOrCreate(conf=configuration)

#depricated
#spark = SparkSession.builder.master('local').appName('states').getOrCreate()
#sc = spark.sparkContext

sqlCon = SQLContext(sc)

/usr/local/lib/python3.10/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [28]:
myStates = 'Alabama California Colorado Arkansas'
rdd = sc.parallelize(myStates.split(' '))

rowdata = rdd.map(lambda x: Row(StateName = x))
print(rowdata.collect())

[Row(StateName='Alabama'), Row(StateName='California'), Row(StateName='Colorado'), Row(StateName='Arkansas')]


In [29]:
df = sqlCon.createDataFrame(rowdata)
df.show()

+----------+
| StateName|
+----------+
|   Alabama|
|California|
|  Colorado|
|  Arkansas|
+----------+



In [34]:
df.createOrReplaceTempView('states')
#mySQL = sqlCon.sql("select StateName from States where StateName like 'Al%'")
mySQL = sqlCon.sql('select StateName from States where StateName like "Al%"')
mySQL.show()

+---------+
|StateName|
+---------+
|  Alabama|
+---------+



In [36]:
myCount = sqlCon.sql('select count(*) as StateCount from States')
myCount.show()

+----------+
|StateCount|
+----------+
|         4|
+----------+

